# DECOMP — interactive exploration

V1-vs-cerebellum movement signal decomposition on the IBL Brain-Wide Map.

This notebook walks through every stage of the pipeline implemented in `src/decomp/`. It is
complementary to `run_all.py` (the artifact-producing entry point); use this notebook for
diagnostics, parameter sweeps, and per-session sanity checks.

All design decisions and the supporting deep-research are logged at
`scratch/2026-05-04-v1-cb-movement-decomposition/`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from one.api import ONE
from brainwidemap import bwm_query

from decomp.pipeline.stage_data import load_and_bin, select_sessions
from decomp.pipeline.stage_glm import run_session_glm
from decomp.pipeline.stage_svca import run_session_svca
from decomp.pipeline.stage_cca import run_session_cca
from decomp.viz.figures import (
    fig01_glm_dr2_per_region, fig02_svca_reliability,
    fig03_cca_canonical_correlations, fig04_pcca_vs_cca,
)

CACHE = Path('data/cache')
OUT = Path('outputs')

## 1. Session selection (Gate-1 fallback ladder)

In [ ]:
one = ONE(base_url='https://openalyx.internationalbrainlab.org', password='international', silent=True)
plan = select_sessions(one, target_n=3, freeze='2023_12_bwm_release', cache_dir=CACHE)
print(plan.strategy_note)
plan.coverage.head()

## 2. Load + bin one session

In [ ]:
bwm = bwm_query(one)
eid_to_pids = bwm.groupby('eid')['pid'].apply(list).to_dict()
eid = plan.eids[0]
pids = eid_to_pids[eid]
sd, binned = load_and_bin(one, eid, pids, rois=plan.rois_used, cache_dir=CACHE)
print(f"{eid}: T={len(binned.bin_centers)} bins, ROIs={list(binned.spikes_by_roi)}")
for roi, mat in binned.spikes_by_roi.items():
    print(f"  {roi}: {mat.shape[0]} units")

## 3. Per-region GLM ΔR² (sanity check vs IBL BWM / Wang 2026)

In [ ]:
glm_df = run_session_glm(sd, binned, rois=plan.rois_used, cache_dir=CACHE)
glm_df.groupby('region')[['dR2_movement','dR2_stim','dR2_choice']].median()

In [ ]:
fig01_glm_dr2_per_region(glm_df, OUT)

## 4. Within-region SVCA

In [ ]:
svca = run_session_svca(binned, cache_dir=CACHE)
for roi, res in svca.items():
    print(f"{roi}: k_reliable={res.k_reliable}, top reliability={res.reliability[:5]}")
fig02_svca_reliability(svca, OUT)

## 5. Pairwise CCA + pCCA — the answer

In [ ]:
cca_df = run_session_cca(binned, svca, cache_dir=CACHE,
                          n_components=8, n_surrogates=100)
fig03_cca_canonical_correlations(cca_df, OUT)
fig04_pcca_vs_cca(cca_df, OUT)
cca_df.groupby(['pair_a','pair_b'])[['rho_cca','rho_pcca']].sum()